# Task #46 — Bỏ giá trị thiếu, chia train/test 80/20 stratified (Story #8, giai đoạn 2)

Đọc `data/processed/orders_features.csv` (Task #44/#45), áp dụng Phương án A (loại `is_delayed=NA`, quyết định Story #6), bỏ các đơn còn thiếu giá trị đặc trưng (số lượng nhỏ, đã đo ở bước đề xuất — không impute), rồi chia train/test 80/20 stratified theo `is_delayed`.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/orders_features.csv", low_memory=False)

df["is_delayed"] = df["is_delayed"].astype("boolean")

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

print(df.shape)
df.dtypes.value_counts()

(99441, 77)


bool       50
float64    18
boolean     7
str         1
int64       1
Name: count, dtype: int64

## 1. Phương án A — loại `is_delayed=NA`

In [2]:
dfa = df[df["is_delayed"].notna()].copy()
print("Sau Phuong an A:", len(dfa), "don")
print(dfa["is_delayed"].value_counts(normalize=True))

Sau Phuong an A: 96476 don
is_delayed
False    0.918871
True     0.081129
Name: proportion, dtype: Float64


## 2. Bỏ đơn còn thiếu giá trị đặc trưng

30 đơn (0,03%), đã đo ở bước đề xuất phương án — không đủ lớn để cần chiến lược impute riêng.

In [3]:
feature_cols = [c for c in dfa.columns if c not in ("order_id", "is_delayed")]
any_missing = dfa[feature_cols].isna().any(axis=1)
print("So don con thieu:", any_missing.sum())

dfa_clean = dfa[~any_missing].copy()
print("Con lai:", len(dfa_clean), "don")

So don con thieu: 31


Con lai: 96445 don


## 3. Chia train/test 80/20, stratified theo `is_delayed`

`random_state=42` để tái lập được.

In [4]:
X = dfa_clean.drop(columns=["is_delayed"])
y = dfa_clean["is_delayed"].astype(bool)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)

print("Train:", X_train.shape, "- ty le tre:", y_train.mean())
print("Test :", X_test.shape, "- ty le tre:", y_test.mean())

Train: (77156, 76) - ty le tre: 0.08114728601793768
Test : (19289, 76) - ty le tre: 0.08113432526310332
